# SpaceX Falcon 9 - Data Collection with REST API
This notebook collects past SpaceX launch records, expands related rocket/payload/launchpad/core information, filters to Falcon 9, and prepares a tabular dataset for the capstone.


In [ ]:
import requests
import pandas as pd
import numpy as np

spacex_url = 'https://api.spacexdata.com/v4/launches/past'
response = requests.get(spacex_url)
response.raise_for_status()
data = pd.json_normalize(response.json())
data.head()


## Expand API identifiers
The launch response contains IDs for rockets, payloads, launchpads and cores. The following helper function resolves those IDs through the related API endpoints.


In [ ]:
def api_get(endpoint, item_id):
    r = requests.get(f'https://api.spacexdata.com/v4/{endpoint}/{item_id}')
    r.raise_for_status()
    return r.json()

records = []
for launch in response.json():
    rocket = api_get('rockets', launch['rocket'])
    if rocket.get('name') != 'Falcon 9':
        continue
    payload = api_get('payloads', launch['payloads'][0]) if launch.get('payloads') else {}
    pad = api_get('launchpads', launch['launchpad']) if launch.get('launchpad') else {}
    core = launch.get('cores', [{}])[0]
    records.append({
        'FlightNumber': launch.get('flight_number'),
        'Date': launch.get('date_utc'),
        'BoosterVersion': rocket.get('name'),
        'PayloadMass': payload.get('mass_kg'),
        'Orbit': payload.get('orbit'),
        'LaunchSite': pad.get('name'),
        'Outcome': core.get('landing_success'),
        'Flights': core.get('flight'),
        'GridFins': core.get('gridfins'),
        'Reused': core.get('reused'),
        'Legs': core.get('legs'),
        'LandingPad': core.get('landpad'),
        'Block': core.get('block'),
        'ReusedCount': rocket.get('stages'),
        'Longitude': pad.get('longitude'),
        'Latitude': pad.get('latitude')
    })
api_df = pd.DataFrame(records)
api_df['PayloadMass'] = api_df['PayloadMass'].fillna(api_df['PayloadMass'].mean())
api_df.head()


## Capstone checks
The course API exercise produces a Falcon 9-only dataset after Falcon 1 launches are removed. The collected table is then used by the wrangling and EDA stages.


In [ ]:
print('Falcon 9 records:', len(api_df))
print('Missing landing-pad values:', api_df['LandingPad'].isna().sum())
api_df.to_csv('dataset_part_1.csv', index=False)
